# 06 — Évaluation robuste des modèles de sentiment

**Évaluation uniquement : aucune cellule n'entraîne, ne réenregistre ni ne modifie un modèle.** Les modèles 03, 04 et 05 sont comparés à partir des prédictions du run `20261115T120646Z_7cae2241`.

Ce notebook produit des tableaux, des figures PNG/SVG/PDF et un rapport dans un **nouveau répertoire daté**. Les données brutes, préparées, prédictions sources et notebooks historiques sont vérifiés par empreintes avant et après l'exécution. Les anciens notebooks ne sont jamais exécutés.

Les statistiques sont calculées avec NumPy ; les prédictions proviennent de modèles déjà exécutés sur le cluster Spark. Aucun Spark en mode local n'est utilisé. Les consultations HDFS sont facultatives et strictement en lecture seule.

Le test AfriSenti ayant déjà influencé le développement, tous les résultats sont **exploratoires**. Les intervalles bootstrap ne rendent pas ce test indépendant et ne corrigent pas les comparaisons multiples.

**Exécution :** kernel Python avec NumPy et Matplotlib, ou depuis la racine du dépôt :

```powershell
powershell.exe -NoProfile -ExecutionPolicy Bypass -File .\scripts\Invoke-EvaluationNotebook.ps1
```

Le runner sauvegarde une copie réellement exécutée du notebook dans le répertoire du run. Le notebook source reste inchangé. Les dépendances existantes ne sont pas réinstallées automatiquement.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from itertools import combinations
import ast
import csv
import hashlib
import html
import importlib.metadata
import io
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import uuid
import zlib

import numpy as np

SEED = 20261115
BOOTSTRAPS = 2000
EXPECTED_TEST_ROWS = 2961
MIN_GROUP_SIZE = 100        # Reporting policy, not a power calculation.
MIN_CLASS_SUPPORT = 30
CHECK_HDFS = True           # Missing Docker/HDFS is reported, not silently replaced.
SCAN_NEAR_DUPLICATES = True
REFERENCE_RUN = "20261115T120646Z_7cae2241"
LABELS = ["negative", "neutral", "positive"]
MODELS = ["03", "04", "05"]
PREDICTION_PREFIX = {"03": "notebook03", "04": "notebook04", "05": "new"}

candidates = [Path.cwd()] + list(Path.cwd().parents)
ROOT = next((p for p in candidates if (p / "notebooks/06_sentiment_robust_evaluation.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the repository root or its notebooks directory.")
SOURCE_NOTEBOOK = ROOT / "notebooks/06_sentiment_robust_evaluation.ipynb"
INPUT_DIR = ROOT / "data/processed/afrisenti/v1"
RAW_DIR = ROOT / "data/raw/afrisenti/v1"
PRED_DIR = INPUT_DIR.parent / "diagnostics" / REFERENCE_RUN
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
OUT_DIR = INPUT_DIR.parent / "evaluations" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=False)
for folder in ("tables", "figures", "private_review", "_matplotlib"):
    (OUT_DIR / folder).mkdir()
os.environ["MPLCONFIGDIR"] = str(OUT_DIR / "_matplotlib")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

STARTED = time.monotonic()
NOTEBOOK_IMAGES = []
ARTIFACTS = []
INPUT_HASHES = {}
CHECKS = {}
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def guard(path):
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError("Required artifact missing: " + str(path))
    INPUT_HASHES[str(path.relative_to(ROOT))] = sha256(path)
    return path

def output_path(relative):
    path = OUT_DIR / relative
    if path.exists():
        raise FileExistsError("Refusing to overwrite an evaluation artifact: " + str(path))
    return path

def write_json(relative, value):
    path = output_path(relative)
    with path.open("x", encoding="utf-8", newline="\n") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False)
        handle.write("\n")
    ARTIFACTS.append(path)
    return path

def write_csv(relative, rows, fields):
    path = output_path(relative)
    with path.open("x", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="raise")
        writer.writeheader()
        writer.writerows(rows)
    ARTIFACTS.append(path)
    return path

def read_csv(path):
    with guard(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, strict=True)
        if not reader.fieldnames or len(reader.fieldnames) != len(set(reader.fieldnames)):
            raise ValueError("Missing or duplicate CSV columns: " + str(path))
        rows = list(reader)
    if any(None in row or any(v is None for v in row.values()) for row in rows):
        raise ValueError("Malformed CSV row: " + str(path))
    return rows

def numeric_label(value):
    number = float(value)
    if not math.isfinite(number) or number not in (0.0, 1.0, 2.0):
        raise ValueError("Invalid label/prediction: " + str(value))
    return int(number)

git_head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True)
git_status = subprocess.run(["git", "--no-optional-locks", "status", "--porcelain"],
                            cwd=ROOT, capture_output=True, text=True)
VERSIONS = {"python": sys.version.split()[0], "numpy": np.__version__,
            "matplotlib": matplotlib.__version__, "unicode_database": unicodedata.unidata_version,
            "git_commit": git_head.stdout.strip() if git_head.returncode == 0 else None,
            "git_dirty_at_start": bool(git_status.stdout.strip()) if git_status.returncode == 0 else None}
guard(SOURCE_NOTEBOOK)
print("Evaluation run:", RUN_ID)
print("Runtime:", VERSIONS)
print("Output:", OUT_DIR)

## 1. Comparabilité et provenance

Les trois fichiers de prédictions doivent contenir exactement les identifiants du même test de 2 961 lignes, sans doublon, et les mêmes labels. L'ordre des lignes est ignoré : l'alignement se fait explicitement par identifiant.

Les empreintes des trois splits préparés sont comparées au manifeste et au rapport d'origine. Les scores historiques servent de contrôle de non-régression du calcul, pas de cible d'optimisation. Les dates et versions des modèles sont extraites des artefacts existants.

In [ ]:
reference = json.loads(guard(PRED_DIR / "report.json").read_text(encoding="utf-8"))
prepared_manifest = json.loads(guard(INPUT_DIR / "audit.json").read_text(encoding="utf-8"))
raw_manifest = json.loads(guard(RAW_DIR / "manifest.json").read_text(encoding="utf-8"))
assert reference["status"] == "completed" and reference["run_id"] == REFERENCE_RUN
assert reference["historical_test_already_consulted"] is True
DATA = {}
for split in ("train", "dev", "test"):
    path = INPUT_DIR / (split + ".csv")
    rows = read_csv(path)
    expected = reference["input_audit"][split]
    assert sha256(path) == expected["sha256"] == prepared_manifest["splits"][split]["sha256"]
    assert len(rows) == expected["rows"] == len({r["record_id"] for r in rows})
    assert all(numeric_label(r["label"]) == LABELS.index(r["sentiment"]) for r in rows)
    assert all(r["comment_text"].strip() and r["source_split"] == split for r in rows)
    DATA[split] = rows
    raw_path = guard(RAW_DIR / (split + ".tsv"))
    assert sha256(raw_path) == raw_manifest["files"][split + ".tsv"]["sha256"]

test_by_id = {r["record_id"]: r for r in DATA["test"]}
IDS = sorted(test_by_id)
assert len(IDS) == EXPECTED_TEST_ROWS
Y = np.array([numeric_label(test_by_id[k]["label"]) for k in IDS], dtype=np.int64)
PRED = {}
for model in MODELS:
    rows = read_csv(PRED_DIR / (PREDICTION_PREFIX[model] + "_test_predictions.csv"))
    assert len(rows) == len(IDS) == len({r["record_id"] for r in rows})
    by_id = {r["record_id"]: r for r in rows}
    assert set(by_id) == set(IDS), "Prediction ID mismatch: " + model
    assert all(numeric_label(by_id[k]["label"]) == numeric_label(test_by_id[k]["label"]) for k in IDS)
    PRED[model] = np.array([numeric_label(by_id[k]["prediction"]) for k in IDS], dtype=np.int64)
    # Optional development predictions are fingerprinted but not used for tuning.
    optional = PRED_DIR / (PREDICTION_PREFIX[model] + "_dev_predictions.csv")
    if optional.is_file():
        guard(optional)

SOURCE = {}
for number, name in [(1,"01_prepare_afrisenti"), (2,"02_train_sentiment_spark"),
                     (3,"03_sentiment_experiments"), (4,"04_sentiment_tfidf_normalized"),
                     (5,"05_sentiment_script_aware")]:
    path = guard(ROOT / "notebooks" / (name + ".ipynb"))
    nb = json.loads(path.read_text(encoding="utf-8"))
    SOURCE[number] = {"path": str(path.relative_to(ROOT)), "notebook": nb,
                      "code": "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")}
    ast.parse(SOURCE[number]["code"])
CHECKS["same_test_identifiers_and_labels"] = {"status": "passed", "rows": len(IDS),
    "ordered_ids_sha256": hashlib.sha256("\n".join(IDS).encode()).hexdigest()}
CHECKS["notebook05_matches_recorded_source"] = {
    "status": "passed" if sha256(ROOT / SOURCE[5]["path"]) == reference["notebook_sha256"] else "warning",
    "expected_sha256": reference["notebook_sha256"], "observed_sha256": sha256(ROOT / SOURCE[5]["path"])}
CHECKS["dataset_source"] = {"source_ref": raw_manifest.get("source_ref"),
    "immutable_git_revision_pinned": bool(raw_manifest.get("source_commit")),
    "note": "Hashes match the existing local manifests; upstream is not downloaded again."}
assignments = read_csv(PRED_DIR / "inner_split.csv")
assert sha256(PRED_DIR / "inner_split.csv") == reference["inner_split"]["assignment_sha256"]
ROLES = {r["record_id"]: r["role"] for r in assignments}
assert len(ROLES) == len(assignments) == len(DATA["train"])
assert set(ROLES) == {r["record_id"] for r in DATA["train"]}
assert set(ROLES.values()) == {"fit", "validation"}
guard(PRED_DIR / "selection_locked.json")
guard(PRED_DIR / "selection_protocol.json")
print("Aligned predictions:", {model: len(p) for model, p in PRED.items()})
print("Dataset rows:", {s: len(rows) for s, rows in DATA.items()})

## 2. Écritures, proxy Arabizi et définitions des métriques

La détection d'écriture travaille sur une **copie d'analyse** où les mentions et URL sont neutralisées. Elle compte uniquement des lettres Unicode identifiées comme arabes ou latines. Elle distingue arabe seul, latin seul, mixte et autres. Le texte fourni au modèle et les prédictions ne sont jamais modifiés.

Le **proxy Arabizi numérique v1**, conservé pour comparer avec l'audit, exige un texte latin ou mixte et un token contenant à la fois des lettres ASCII et un chiffre parmi `2,3,5,6,7,8,9`. Il peut manquer l'Arabizi sans chiffres et détecter à tort des termes comme `COVID19`. Ce proxy n'est **jamais une annotation linguistique vérifiée**. Le groupe proxy recoupe les groupes d'écriture ; il ne faut pas additionner leurs effectifs.

Le macro-F1 est la moyenne sur les trois labels fixes, avec F1 nul lorsque précision et rappel sont tous deux indéfinis. Un groupe vide reçoit des métriques non évaluables. Une classe absente ou un groupe petit est signalé. Les seuils de prudence (100 textes, 30 exemples par classe) sont une politique de présentation, pas une preuve de puissance statistique.

Le rappel négatif est accompagné d'un intervalle de Wilson à 95 %. Les autres intervalles et différences sont calculés par bootstrap.

In [ ]:
NEUTRALIZE = re.compile(r"https?://\S+|www\.\S+|@[\w]+", re.IGNORECASE)
def analysis_body(text):
    return NEUTRALIZE.sub(" ", text)

def writing_system(text):
    body = analysis_body(text)
    arabic = any(c.isalpha() and "ARABIC" in unicodedata.name(c, "") for c in body)
    latin = any(c.isalpha() and "LATIN" in unicodedata.name(c, "") for c in body)
    return "mixed" if arabic and latin else "arabic" if arabic else "latin" if latin else "other"

def arabizi_digit_proxy(text):
    if writing_system(text) not in ("latin", "mixed"):
        return False
    tokens = re.findall(r"[A-Za-z0-9]+", analysis_body(text))
    return any(re.search("[A-Za-z]", t) and re.search("[2356789]", t) for t in tokens)

def confusion(y, pred):
    assert len(y) == len(pred)
    return np.bincount(y * 3 + pred, minlength=9).reshape(3, 3)

def metrics(y, pred):
    matrix = confusion(y, pred)
    support, guessed = matrix.sum(axis=1), matrix.sum(axis=0)
    tp = np.diag(matrix)
    precision = np.divide(tp, guessed, out=np.zeros(3), where=guessed > 0)
    recall = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    f1 = np.divide(2 * tp, support + guessed, out=np.zeros(3), where=(support + guessed) > 0)
    return {"n": len(y), "accuracy": float(tp.sum()/len(y)) if len(y) else None,
            "macro_f1": float(f1.mean()) if len(y) else None,
            "weighted_f1": float(np.dot(f1, support)/len(y)) if len(y) else None,
            "confusion": matrix.tolist(),
            "predicted_counts": {name: int(guessed[i]) for i, name in enumerate(LABELS)},
            "per_class": {name: {"support": int(support[i]), "precision": float(precision[i]),
                                "recall": float(recall[i]), "f1": float(f1[i])}
                          for i, name in enumerate(LABELS)}}

def wilson(successes, total):
    if total == 0:
        return [None, None]
    z = 1.959963984540054
    rate = successes / total
    denominator = 1 + z*z/total
    center = (rate + z*z/(2*total)) / denominator
    half = z*math.sqrt(rate*(1-rate)/total + z*z/(4*total*total))/denominator
    return [max(0.0, center-half), min(1.0, center+half)]

def group_warnings(y):
    support = np.bincount(y, minlength=3)
    flags = []
    if not len(y):
        return ["empty_group"]
    if len(y) < MIN_GROUP_SIZE:
        flags.append("small_group_n_lt_" + str(MIN_GROUP_SIZE))
    if (support == 0).any():
        flags.append("missing_true_class")
    if (support < MIN_CLASS_SUPPORT).any():
        flags.append("class_support_lt_" + str(MIN_CLASS_SUPPORT))
    return flags

# Small independent checks of the new evaluator.
assert metrics(np.array([0,1,2]), np.array([0,1,2]))["macro_f1"] == 1.0
assert metrics(np.array([0,1,2]), np.array([1,2,0]))["macro_f1"] == 0.0
assert metrics(np.array([], dtype=int), np.array([], dtype=int))["macro_f1"] is None
assert abs(metrics(np.array([0,1]), np.array([0,1]))["macro_f1"] - 2/3) < 1e-12
assert wilson(0, 0) == [None, None] and wilson(0, 10)[0] < 1e-12
assert writing_system("@user \u0633\u0644\u0627\u0645 https://example.com/en") == "arabic"
assert writing_system("bonjour \u00e9cole") == "latin"
assert writing_system("\u0633\u0644\u0627\u0645 hello") == "mixed"
assert writing_system("123 \U0001f600 @user https://example.com") == "other"
assert arabizi_digit_proxy("ma3jbnich hadchi") and not arabizi_digit_proxy("bonjour 2026")
assert arabizi_digit_proxy("COVID19")  # Documented false-positive example; not a language label.

SCRIPTS = np.array([writing_system(test_by_id[k]["comment_text"]) for k in IDS])
PROXY = np.array([arabizi_digit_proxy(test_by_id[k]["comment_text"]) for k in IDS])
GROUPS = {"all": np.arange(len(IDS))}
for name in ("arabic", "latin", "mixed", "other"):
    GROUPS[name] = np.flatnonzero(SCRIPTS == name)
GROUPS["arabizi_proxy"] = np.flatnonzero(PROXY)
GROUP_NAMES = {"all": "Ensemble", "arabic": "Arabe seul", "latin": "Latin seul",
               "mixed": "Mixte arabe/latin", "other": "Autres",
               "arabizi_proxy": "Proxy Arabizi*"}
RESULTS = {g: {m: metrics(Y[idx], PRED[m][idx]) for m in MODELS} for g, idx in GROUPS.items()}
for model in MODELS:
    old = reference["evaluation"][PREDICTION_PREFIX[model]]["test"]["overall"]
    now = RESULTS["all"][model]
    assert now["confusion"] == old["confusion_matrix"]
    assert abs(now["macro_f1"] - old["macro_f1"]) < 1e-12
    assert abs(now["accuracy"] - old["accuracy"]) < 1e-12
    assert now["n"] == EXPECTED_TEST_ROWS
CHECKS["metric_regression_checks"] = {"status": "passed", "reference_run": REFERENCE_RUN,
    "confusion_and_accuracy_and_macro_f1_match": True}
SCRIPT_COUNTS = {s: dict(Counter(writing_system(r["comment_text"]) for r in rows))
                 for s, rows in DATA.items()}
print("Observed script counts:", SCRIPT_COUNTS)
print("Test macro-F1:", {m: round(RESULTS["all"][m]["macro_f1"], 8) for m in MODELS})

## 3. Incertitude et comparaisons appariées

Pour chaque groupe, les 2 000 rééchantillonnages sélectionnent les **mêmes indices pour les trois modèles**. Le groupe global utilise la graine historique 20261115 ; les autres groupes ont une graine dérivée de leur nom par SHA-256.

On calcule les intervalles percentiles à 95 % du macro-F1 et des différences 05−03 et 05−04. Un intervalle contenant zéro ne démontre pas de supériorité. Pour le rappel négatif, les tirages sans négatif sont invalides ; leur nombre est publié, et l'intervalle de différence est masqué si moins de 95 % des tirages sont valides.

Hypothèses et limites : rééchantillonnage de commentaires supposés indépendants, aucune correction de multiplicité, absence d'identifiants d'auteur/conversation permettant un bootstrap groupé. Les statistiques restent exploratoires.

In [ ]:
def fast_scores(y, pred):
    matrix = confusion(y, pred)
    support, guessed = matrix.sum(axis=1), matrix.sum(axis=0)
    diag = np.diag(matrix)
    f1 = np.divide(2*diag, support+guessed, out=np.zeros(3), where=(support+guessed)>0)
    return (float(f1.mean()), float(diag.sum()/len(y)),
            float(matrix[0,0]/support[0]) if support[0] else np.nan)

def bootstrap_arrays(y, predictions, seed, repetitions):
    rng = np.random.default_rng(seed)
    samples = {m: np.empty((repetitions, 3)) for m in predictions}
    for b in range(repetitions):
        positions = rng.integers(0, len(y), size=len(y))
        for m, predicted in predictions.items():
            samples[m][b] = fast_scores(y[positions], predicted[positions])
    return samples

def percentile(values, required_fraction=0.95):
    valid = values[np.isfinite(values)]
    if len(valid) < len(values)*required_fraction:
        return [None, None]
    return [float(x) for x in np.quantile(valid, [0.025, 0.975])]

fixture_y = np.tile([0,1,2], 10)
fixture = bootstrap_arrays(fixture_y, {"03": fixture_y, "05": fixture_y}, 123, 20)
assert np.array_equal(fixture["03"], fixture["05"])
assert percentile(fixture["05"][:,0]-fixture["03"][:,0]) == [0.0, 0.0]

CI = {}
COMPARISONS = []
GLOBAL_BOOTSTRAP = []
for group, indices in GROUPS.items():
    CI[group] = {}
    if not len(indices):
        for m in MODELS:
            CI[group][m] = {"macro_f1": [None,None], "accuracy": [None,None],
                           "negative_recall": [None,None]}
        continue
    seed = SEED if group == "all" else SEED + int(hashlib.sha256(group.encode()).hexdigest()[:8], 16)
    samples = bootstrap_arrays(Y[indices], {m:PRED[m][indices] for m in MODELS}, seed, BOOTSTRAPS)
    warnings = group_warnings(Y[indices])
    for model in MODELS:
        met = RESULTS[group][model]
        nneg = met["per_class"]["negative"]["support"]
        CI[group][model] = {
            "macro_f1": percentile(samples[model][:,0]),
            "accuracy": wilson(sum(met["confusion"][k][k] for k in range(3)), len(indices)),
            "negative_recall": wilson(met["confusion"][0][0], nneg)}
    for baseline in ("03", "04"):
        deltas = samples["05"] - samples[baseline]
        new, old = RESULTS[group]["05"], RESULTS[group][baseline]
        nneg = new["per_class"]["negative"]["support"]
        row = {"group": group, "comparison": "05-" + baseline, "n": len(indices),
               "negative_support": nneg, "bootstrap_seed": seed, "bootstrap_repetitions": BOOTSTRAPS,
               "delta_macro_f1": new["macro_f1"]-old["macro_f1"],
               "macro_ci_low": percentile(deltas[:,0])[0], "macro_ci_high": percentile(deltas[:,0])[1],
               "delta_accuracy": new["accuracy"]-old["accuracy"],
               "accuracy_ci_low": percentile(deltas[:,1])[0], "accuracy_ci_high": percentile(deltas[:,1])[1],
               "delta_negative_recall": new["per_class"]["negative"]["recall"]-old["per_class"]["negative"]["recall"] if nneg else None,
               "negative_ci_low": percentile(deltas[:,2])[0], "negative_ci_high": percentile(deltas[:,2])[1],
               "negative_valid_bootstrap_replicates": int(np.isfinite(deltas[:,2]).sum()),
               "warnings": ";".join(warnings), "exploratory": True}
        COMPARISONS.append(row)
        if group == "all":
            for b, values in enumerate(deltas):
                GLOBAL_BOOTSTRAP.append({"comparison": "05-"+baseline, "replicate": b,
                    "delta_macro_f1": float(values[0]), "delta_accuracy": float(values[1]),
                    "delta_negative_recall": float(values[2])})
    print("Bootstrap completed:", group, "n=", len(indices))

# Regression check against the recorded paired bootstrap, using identical sorted IDs.
for baseline in ("03","04"):
    row = next(r for r in COMPARISONS if r["group"] == "all" and r["comparison"] == "05-"+baseline)
    expected = reference["paired_bootstrap"][PREDICTION_PREFIX[baseline]]
    assert abs(row["delta_macro_f1"] - expected["delta_macro_f1"]) < 1e-12
    assert np.allclose([row["macro_ci_low"],row["macro_ci_high"]], expected["percentile_95"], atol=1e-12)
CHECKS["paired_bootstrap_regression"] = {"status":"passed", "replicates":BOOTSTRAPS, "seed":SEED}

metric_rows, class_rows, confusion_rows, distribution_rows = [], [], [], []
for group, models in RESULTS.items():
    for model, met in models.items():
        low, high = CI[group][model]["macro_f1"]
        rlo, rhi = CI[group][model]["negative_recall"]
        metric_rows.append({"group":group, "model":model, "n":met["n"], "accuracy":met["accuracy"],
            "macro_f1":met["macro_f1"], "macro_ci_low":low, "macro_ci_high":high,
            "negative_support":met["per_class"]["negative"]["support"],
            "negative_recall":met["per_class"]["negative"]["recall"] if met["per_class"]["negative"]["support"] else None,
            "negative_ci_low":rlo, "negative_ci_high":rhi,
            "warnings":";".join(group_warnings(Y[GROUPS[group]]))})
        for k, name in enumerate(LABELS):
            class_rows.append(dict(group=group, model=model, sentiment=name, **met["per_class"][name]))
            distribution_rows.append({"group":group,"model":model,"predicted_class":name,
                "count":met["predicted_counts"][name],
                "proportion":met["predicted_counts"][name]/met["n"] if met["n"] else None})
            for j, predicted in enumerate(LABELS):
                confusion_rows.append({"group":group,"model":model,"true_class":name,
                    "predicted_class":predicted,"count":met["confusion"][k][j]})
write_csv("tables/metrics_by_group.csv",metric_rows,list(metric_rows[0]))
write_csv("tables/per_class_metrics.csv",class_rows,list(class_rows[0]))
write_csv("tables/confusion_matrices.csv",confusion_rows,list(confusion_rows[0]))
write_csv("tables/prediction_distribution.csv",distribution_rows,list(distribution_rows[0]))
write_csv("tables/paired_comparisons.csv",COMPARISONS,list(COMPARISONS[0]))
write_csv("tables/paired_bootstrap_global.csv",GLOBAL_BOOTSTRAP,list(GLOBAL_BOOTSTRAP[0]))

## 4. Où les modèles sont en désaccord

Les désaccords sont répartis en corrections apportées par 05, nouvelles erreurs de 05, et erreurs différentes des deux modèles. Cette décomposition explique la variation d'accuracy, mais **ne s'additionne pas directement en macro-F1**.

Les tableaux publics contiennent des identifiants et des comptes. Les textes et exemples à relire sont conservés uniquement dans `private_review/`, sous le répertoire local ignoré par Git. Les causes proposées doivent être annotées par un humain : présence d'une négation ou d'une répétition n'est pas une preuve causale.

In [ ]:
ERROR_COUNTS, DISAGREEMENTS, EXAMPLES = [], [], []
for model in MODELS:
    for group, indices in GROUPS.items():
        met = RESULTS[group][model]
        for i, actual in enumerate(LABELS):
            for j, predicted in enumerate(LABELS):
                if i != j and met["confusion"][i][j]:
                    ERROR_COUNTS.append({"model":model, "group":group, "true_class":actual,
                        "predicted_class":predicted, "count":met["confusion"][i][j]})
ERROR_COUNTS.sort(key=lambda r:(r["model"],r["group"],-r["count"],r["true_class"]))
write_csv("tables/frequent_errors.csv",ERROR_COUNTS,list(ERROR_COUNTS[0]))

DISAGREEMENT_SUMMARIES = []
for baseline in ("03","04"):
    different = PRED["05"] != PRED[baseline]
    for group, indices in GROUPS.items():
        mask = different[indices]
        ids = indices[mask]
        wins = int(np.sum((PRED["05"][ids]==Y[ids]) & (PRED[baseline][ids]!=Y[ids])))
        losses = int(np.sum((PRED["05"][ids]!=Y[ids]) & (PRED[baseline][ids]==Y[ids])))
        both_wrong = int(np.sum((PRED["05"][ids]!=Y[ids]) & (PRED[baseline][ids]!=Y[ids])))
        assert wins+losses+both_wrong == len(ids)
        assert int(np.sum(PRED["05"][indices]==Y[indices])-np.sum(PRED[baseline][indices]==Y[indices])) == wins-losses
        DISAGREEMENT_SUMMARIES.append({"comparison":"05-"+baseline,"group":group,
            "n":len(indices),"disagreements":len(ids),"corrected_by_05":wins,
            "new_errors_by_05":losses,"both_wrong_different":both_wrong,"net_correct":wins-losses})
    for i in np.flatnonzero(different):
        kind = "corrected_by_05" if PRED["05"][i] == Y[i] else "new_error_by_05" if PRED[baseline][i] == Y[i] else "both_wrong_different"
        row = {"comparison":"05-"+baseline, "record_id":IDS[i], "true_class":LABELS[Y[i]],
               "baseline_prediction":LABELS[PRED[baseline][i]], "prediction_05":LABELS[PRED["05"][i]],
               "writing_script":str(SCRIPTS[i]), "arabizi_proxy_unverified":bool(PROXY[i]), "change":kind}
        DISAGREEMENTS.append(row)
        original = test_by_id[IDS[i]]["comment_text"]
        EXAMPLES.append(dict(row, comment_text_original=original,
            has_mention=bool(re.search(r"@[\w]+",original)),
            has_url=bool(re.search(r"https?://|www\.",original,re.I)),
            repeated_letters=bool(re.search(r"([a-z\u0621-\u064a])\1{2,}",original,re.I)),
            human_language="", human_error_cause="", human_notes=""))
write_csv("tables/disagreement_summary.csv",DISAGREEMENT_SUMMARIES,list(DISAGREEMENT_SUMMARIES[0]))
write_csv("tables/disagreement_identifiers.csv",DISAGREEMENTS,list(DISAGREEMENTS[0]))
write_csv("private_review/disagreements_with_text.csv",EXAMPLES,list(EXAMPLES[0]))

# Fixed small review panel: hashed ordering within comparison / direction / script.
review_groups = defaultdict(list)
for row in EXAMPLES:
    review_groups[(row["comparison"],row["change"],row["writing_script"])].append(row)
REVIEW = []
for key in sorted(review_groups):
    ordered = sorted(review_groups[key], key=lambda r: hashlib.sha256((str(SEED)+r["record_id"]).encode()).hexdigest())
    REVIEW.extend(ordered[:5])
write_csv("private_review/stratified_review_panel.csv",REVIEW,list(EXAMPLES[0]))

# Aggregate prediction transitions; textual markers are not proven causes.
transition_counts=Counter((r["comparison"],r["writing_script"],r["true_class"],
    r["baseline_prediction"],r["prediction_05"],r["change"]) for r in DISAGREEMENTS)
TRANSITIONS=[dict(zip(("comparison","writing_script","true_class","baseline_prediction",
    "prediction_05","change"),key),count=count) for key,count in
    sorted(transition_counts.items(),key=lambda item:(item[0][0],-item[1],item[0]))]
assert sum(r["count"] for r in TRANSITIONS)==len(DISAGREEMENTS)
write_csv("tables/disagreement_transitions.csv",TRANSITIONS,list(TRANSITIONS[0]))
CLASS_CHANGES=[]
for comparison in ("05-03","05-04"):
    for label in LABELS:
        subset=[r for r in DISAGREEMENTS if r["comparison"]==comparison and r["true_class"]==label]
        tally=Counter(r["change"] for r in subset)
        CLASS_CHANGES.append({"comparison":comparison,"true_class":label,
            "disagreements":len(subset),"corrected_by_05":tally["corrected_by_05"],
            "new_errors_by_05":tally["new_error_by_05"],
            "both_wrong_different":tally["both_wrong_different"],
            "net_correct":tally["corrected_by_05"]-tally["new_error_by_05"]})
write_csv("tables/disagreement_by_class.csv",CLASS_CHANGES,list(CLASS_CHANGES[0]))

# Illustrations read during this evaluation, NOT independent human annotations.
# Review hypotheses, never causal model explanations or replacement labels.
CASE_NOTES=[
    ("05-03","afrisenti:ary:test:30bc04b472f0b0da",
     "Positif corrigé : salutation de soutien et vocabulaire de désordre coexistent. "
     "La cible du sentiment est une piste de relecture ; aucune attribution causale testée."),
    ("05-03","afrisenti:ary:test:0118d834613cf88b",
     "Nouvelle erreur sur un résultat sportif annoté neutre. "
     "Le passage à négatif illustre une polarité attribuée à un énoncé factuel ; cause inconnue."),
    ("05-03","afrisenti:ary:test:410265c9d6956ec7",
     "Négatif perdu vers neutre par 05. Le texte contient une accusation de mensonge "
     "et des chiffres dans des mots latins ; le proxy Arabizi n'est pas validé humainement."),
    ("05-03","afrisenti:ary:test:145e34b2ce3db2cf",
     "Négatif devenu positif. Le marqueur littéral [[positiveHOTO]] déclenche la composante latine "
     "de la règle : un groupe mixte détecté peut contenir un artefact. Son influence sur la "
     "prédiction n'a pas été testée. Texte et label inchangés."),
    ("05-03","afrisenti:ary:test:d34a7e75acf87e32",
     "Deux erreurs différentes sur une citation sportive annotée neutre. "
     "La distinction entre propos rapporté et sentiment de l'auteur mérite une relecture "
     "du guide, sans requalifier automatiquement le label."),
    ("05-04","afrisenti:ary:test:ae05607629ea8c86",
     "Positif corrigé depuis neutre. Texte latin, emoji, aucun chiffre déclenchant le proxy : "
     "Latin et Arabizi détecté ne sont pas synonymes. Cela ne prouve pas l'absence réelle d'Arabizi."),
    ("05-04","afrisenti:ary:test:49ea1507060af757",
     "Neutre devenu négatif sur un titre interrogatif. "
     "Ce contre-exemple accompagne le gain global : toutes les décisions de 04 ne sont pas améliorées."),
    ("05-04","afrisenti:ary:test:303af7da8ab8d854",
     "Négatif manqué par les deux modèles : neutre pour 04, positif pour 05. "
     "Un changement de prédiction n'est donc pas nécessairement une correction."),
]
CASE_SUMMARIES=[]
case_lines=["# Cas illustratifs : notes de lecture exploratoires","",
    "Cas du panel déterministe, sélection illustrative non représentative. Notes formulées "
    "lors de cette analyse, sans double annotation humaine ni expérience d'attribution. "
    "Labels conservés ; champs d'annotation humaine des CSV laissés vides.",""]
for comparison,record_id,note in CASE_NOTES:
    row=next(r for r in EXAMPLES if r["comparison"]==comparison and r["record_id"]==record_id)
    summary={k:row[k] for k in ("comparison","record_id","true_class","baseline_prediction",
        "prediction_05","writing_script","change")}
    summary["review_hypothesis"]=note
    CASE_SUMMARIES.append(summary)
    case_lines+=["## "+comparison+" · "+record_id,"",
        "Label du corpus : **"+row["true_class"]+"**. Prédictions : "
        +row["baseline_prediction"]+" → "+row["prediction_05"]+".",
        "Écriture détectée : "+row["writing_script"]+" ; proxy non vérifié : "
        +str(row["arabizi_proxy_unverified"])+".","",
        *["> "+line for line in row["comment_text_original"].splitlines()],"",note,""]
case_path=output_path("private_review/disagreement_case_notes.md")
case_path.write_text("\n".join(case_lines),encoding="utf-8")
ARTIFACTS.append(case_path)
write_csv("tables/illustrative_case_notes.csv",CASE_SUMMARIES,list(CASE_SUMMARIES[0]))
for row in DISAGREEMENT_SUMMARIES:
    if row["group"] == "all":
        print("Disagreements:", row)

## 5. Contrôles de reproductibilité sans réexécuter les anciens notebooks

Le contrôle des quasi-doublons utilise des shingles de cinq caractères, une présélection MinHash déterministe (32 fonctions, 8 bandes de 4), puis le Jaccard exact des paires candidates. Le seuil est 0,85 sur les textes normalisés d'au moins 30 caractères. La recherche est **approximative et non exhaustive**. Aucune ligne ni partition n'est changée.

Les sources historiques sont inspectées comme du texte/AST. Les chemins contenant `overwrite()` sont signalés et jamais exécutés. Les consultations HDFS lisent uniquement les métadonnées : ce contrôle ne remplace pas un checksum intégral des poids ni une nouvelle inférence. Le rechargement et la parité des prédictions ont déjà été vérifiés dans l'audit précédent.

La préparation NFKC du notebook 01 doit rester cohérente avec le futur contrat Silver. Le notebook 06 signale la dépendance externe, sans modifier la préparation ni le modèle.

In [ ]:
def audit_canonical(text):
    text = unicodedata.normalize("NFKC", text).casefold()
    text = NEUTRALIZE.sub(" ", text)
    text = re.sub("[\u0640\u064b-\u065f\u0670]", "", text)
    text = re.sub("[\u0622\u0623\u0625\u0671]", "\u0627", text)
    text = re.sub(r"([a-z\u0621-\u064a])\1{2,}", r"\1\1", text)
    return " ".join(text.split())

text_sets = {s:{audit_canonical(r["comment_text"]) for r in rows} for s, rows in DATA.items()}
CHECKS["exact_normalized_overlap"] = {a+"-"+b:len(text_sets[a]&text_sets[b])
    for a,b in combinations(("train","dev","test"),2)}
CHECKS["normalized_duplicates_within_split"] = {s:len(DATA[s])-len(keys) for s,keys in text_sets.items()}
strata = defaultdict(list)
for r in DATA["train"]:
    t = r["comment_text"]
    old_script = "arabic" if len(re.findall("[\u0600-\u06ff]",t))>len(re.findall("[A-Za-z]",t)) else "latin"
    strata[(old_script,r["label"])].append(r)
expected_roles = {}
for rows in strata.values():
    ordered = sorted(rows,key=lambda r:(hashlib.sha256(("lahjalens-v3-seed-20261115:"+r["comment_text"]).encode()).hexdigest(),r["record_id"]))
    for rank,row in enumerate(ordered,1):
        expected_roles[row["record_id"]] = "validation" if rank%5==0 else "fit"
assert expected_roles == ROLES
CHECKS["internal_split_reproduced"] = {"status":"passed", "counts":dict(Counter(ROLES.values())),
    "note":"Original two-script split reproduced, not replaced by the new descriptive groups."}

near_rows = []
if SCAN_NEAR_DUPLICATES:
    items = []
    for split, rows in DATA.items():
        for row in rows:
            text = audit_canonical(row["comment_text"])
            if len(text)>=30:
                items.append({"id":row["record_id"],"part":ROLES[row["record_id"]] if split=="train" else split,
                              "label":row["sentiment"],"text":text})
    rng = random.Random(SEED)
    prime = (1<<61)-1
    coef = [(rng.randrange(1,prime),rng.randrange(prime)) for _ in range(32)]
    buckets, shingles, pairs = defaultdict(list), [], set()
    for i,row in enumerate(items):
        sh = {row["text"][j:j+5] for j in range(len(row["text"])-4)}
        shingles.append(sh)
        values = [zlib.crc32(t.encode()) for t in sh]
        signature = [min((a*x+b)%prime for x in values) for a,b in coef]
        for band in range(8):
            key = (band,tuple(signature[band*4:band*4+4]))
            for previous in buckets[key]:
                if items[previous]["part"] != row["part"]:
                    pairs.add((previous,i))
            buckets[key].append(i)
    for a,b in sorted(pairs):
        similarity = len(shingles[a]&shingles[b])/len(shingles[a]|shingles[b])
        if similarity>=0.85:
            near_rows.append({"id_a":items[a]["id"],"id_b":items[b]["id"],
                "partition_a":items[a]["part"],"partition_b":items[b]["part"],
                "label_a":items[a]["label"],"label_b":items[b]["label"],"jaccard_char5":similarity})
    CHECKS["near_duplicates"] = {"status":"completed","screened_rows":len(items),
        "candidate_pairs":len(pairs),"detected_pairs":len(near_rows),"threshold":0.85,
        "shingle_size":5,"min_text_length":30,"minhash_functions":32,"bands":8,
        "rows_per_band":4,"seed":SEED,"exhaustive":False}
else:
    CHECKS["near_duplicates"] = {"status":"not_executed"}
write_csv("tables/near_duplicates.csv",near_rows,
          ["id_a","id_b","partition_a","partition_b","label_a","label_b","jaccard_char5"])

def source_locations(number, phrase):
    lines = (ROOT/SOURCE[number]["path"]).read_text(encoding="utf-8").splitlines()
    return [i for i,line in enumerate(lines,1) if phrase in line]

overwrite_findings = []
for number, item in SOURCE.items():
    tree = ast.parse(item["code"])
    calls = [node for node in ast.walk(tree) if isinstance(node,ast.Call)
             and isinstance(node.func,ast.Attribute) and node.func.attr=="overwrite"]
    if calls:
        overwrite_findings.append({"notebook":item["path"],"calls":len(calls),
            "json_lines":source_locations(number,"overwrite()"),
            "risk":"parent afrisenti_v1 also contains historical experiments" if number==2 else "overwrite of an existing run path",
            "executed":False})
CHECKS["historical_overwrite_risks"] = overwrite_findings
CHECKS["preprocessing"] = {
    "notebook01_nfkc_present": "NFKC" in SOURCE[1]["code"],
    "notebook05_nfkc_present": "NFKC" in SOURCE[5]["code"],
    "notebook01_removes_all_cf": "!= 'Cf'" in SOURCE[1]["code"],
    "notebook01_location": {"path":SOURCE[1]["path"],"json_lines":source_locations(1,"category(char)")},
    "notebook05_location": {"path":SOURCE[5]["path"],"json_lines":source_locations(5,"def clean_expression")},
    "status":"external_preprocessing_contract_required",
    "note":"Saved predictions are on prepared CSVs; raw-text/Silver equivalence is not proven by this notebook."}

CHECKS["preprocessing"]["by_model"] = {
    "03": {"char_tokens_created_outside_saved_pipeline":
        "return frame.withColumn('char_tokens', char_ngrams" in SOURCE[3]["code"],
        "required_inference_contract":"prepared comment_text plus the original external char_ngrams/add_features function",
        "source_lines":source_locations(3,"return frame.withColumn"),
        "dynamic_equivalence_tested":False},
    "04": {"sql_character_transformers_declared":"lower_text = SQLTransformer" in SOURCE[4]["code"]
           and "char_tokens = SQLTransformer" in SOURCE[4]["code"],
        "required_inference_contract":"prepared CSV text; SQL character feature stages declared in source pipeline",
        "source_lines":source_locations(4,"lower_text = SQLTransformer"),
        "dynamic_equivalence_tested":False},
    "05": {"clean_expression_declared":"def clean_expression" in SOURCE[5]["code"],
        "required_inference_contract":"prepared NFKC/Cf-cleaned text; no proven raw-text equivalence",
        "dynamic_equivalence_tested":False}}
strict_tsv = {}
for split in ("train","dev","test"):
    try:
        with (RAW_DIR/(split+".tsv")).open(encoding="utf-8-sig",newline="") as handle:
            reader = csv.DictReader(handle,delimiter="\t",strict=True)
            count = sum(1 for _ in reader)
        strict_tsv[split] = {"status":"passed","rows":count}
    except csv.Error as exc:
        strict_tsv[split] = {"status":"warning","physical_line":reader.line_num,"error":str(exc)}
CHECKS["raw_tsv_strict_read"] = strict_tsv

# Read literal historical paths without executing the notebook.
literal_paths = {}
for node in ast.walk(ast.parse(SOURCE[5]["code"])):
    if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name):
        name = node.targets[0].id
        if name in ("OLD03_PATH","OLD04_PATH"):
            literal_paths[name] = ast.literal_eval(node.value)
MODEL_PATHS = {"03":literal_paths["OLD03_PATH"],"04":literal_paths["OLD04_PATH"],"05":reference["model_path"]}
MODEL_METADATA = {}
docker = shutil.which("docker")
if not docker and os.name=="nt":
    candidate = Path(os.environ.get("LOCALAPPDATA",""))/"Programs/DockerDesktop/resources/bin/docker.exe"
    if candidate.is_file():
        docker = str(candidate)
def hdfs_read(arguments):
    result = subprocess.run([docker,"compose","exec","-T","namenode","hdfs","dfs"]+arguments,
        cwd=ROOT,capture_output=True,text=True,encoding="utf-8",errors="replace",timeout=25)
    if result.returncode:
        raise RuntimeError(result.stderr.strip()[-500:] or "HDFS command failed")
    return result.stdout
for model,path in MODEL_PATHS.items():
    assert path.startswith("hdfs://namenode:9000/data/lahjalens/models/")
    if not CHECK_HDFS or not docker:
        MODEL_METADATA[model] = {"path":path,"status":"not_executed","reason":"disabled or Docker CLI unavailable"}
        continue
    try:
        payload = hdfs_read(["-cat",path+"/metadata/part-00000"])
        metadata = json.loads(payload)
        info = {"path":path,"status":"metadata_available","spark_version":metadata.get("sparkVersion"),
            "uid":metadata.get("uid"),"timestamp_ms":metadata.get("timestamp"),
            "root_metadata_sha256":hashlib.sha256(payload.encode()).hexdigest(),
            "whole_model_checksum_verified":False,"model_reinference_executed":False}
        if model=="05":
            first_uid = metadata.get("paramMap",{}).get("stageUids",[])[0]
            listing = hdfs_read(["-ls",path+"/stages"])
            stage_path = next(line.split()[-1] for line in listing.splitlines()
                              if line.split() and line.split()[-1].endswith("_"+first_uid))
            stage_payload = hdfs_read(["-cat",stage_path+"/metadata/part-00000"])
            statement = json.loads(stage_payload).get("paramMap",{}).get("statement","")
            info["first_transformer_statement_sha256"] = hashlib.sha256(statement.encode()).hexdigest()
            info["first_transformer_contains_nfkc"] = "nfkc" in statement.lower()
        MODEL_METADATA[model] = info
    except (OSError,RuntimeError,ValueError,IndexError,StopIteration,subprocess.TimeoutExpired) as exc:
        MODEL_METADATA[model] = {"path":path,"status":"unavailable_or_incomplete","error":str(exc)[:500]}
CHECKS["model_metadata"] = MODEL_METADATA
CHECKS["execution_scope"] = {"training_executed":False,"historical_notebook_executed":False,
    "spark_local_fallback":False,"new_model_inference_executed":False,
    "metrics_recomputed_from_saved_predictions":True,"hdfs_integrity_fsck_repeated":False}

hespress_sample = ROOT/"data/processed/hespress/comments_sample.csv"
if hespress_sample.is_file():
    rows = read_csv(hespress_sample)
    CHECKS["hespress_available_sample"] = {"rows":len(rows),"articles":len({r["article_id"] for r in rows}),
        "sentiment_labels_present":sum(bool(r.get("sentiment","").strip()) for r in rows),
        "script_counts":dict(Counter(writing_system(r["comment_text"]) for r in rows)),
        "independent_sentiment_benchmark_ready":False}
else:
    CHECKS["hespress_available_sample"] = {"status":"missing","independent_sentiment_benchmark_ready":False}
print("Reproducibility checks:", json.dumps({
    "exact_overlap":CHECKS["exact_normalized_overlap"],"near_duplicates":CHECKS["near_duplicates"],
    "hdfs":{m:v["status"] for m,v in MODEL_METADATA.items()},
    "hespress":CHECKS["hespress_available_sample"]},ensure_ascii=False))
write_json("reproducibility_checks.json",CHECKS)

## 6. Figures pour le rapport technique

Les figures utilisent les mêmes labels, couleurs et unités. Les fichiers PNG sont exportés à 220 dpi ; les SVG et PDF sont vectoriels. Un PDF rassemble toutes les figures.

Les matrices montrent les effectifs et les pourcentages par classe réelle. Les comparaisons affichent les différences en **points de pourcentage**, avec zéro comme référence. Les groupes fragiles et le caractère heuristique du proxy Arabizi sont explicités.

In [ ]:
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.titlesize":12,
    "axes.labelsize":10,"axes.spines.top":False,"axes.spines.right":False,
    "figure.facecolor":"white","axes.facecolor":"white","savefig.facecolor":"white",
    "svg.fonttype":"none","pdf.fonttype":42})
COLORS = {"03":"#0072B2","04":"#737373","05":"#D55E00"}
CLASS_COLORS = ["#D55E00","#747C87","#0072B2"]
SCRIPT_COLORS = ["#0072B2","#E69F00","#009E73","#747C87"]
FR_LABELS = ["Négatif","Neutre","Positif"]
FIGURE_FILES = []
def save_figure(fig, name, pdf):
    for extension in ("png","svg","pdf"):
        path = output_path("figures/"+name+"."+extension)
        fig.savefig(path,dpi=220,bbox_inches="tight")
        ARTIFACTS.append(path)
        if extension=="png":
            NOTEBOOK_IMAGES.append(str(path))
            FIGURE_FILES.append(path)
    pdf.savefig(fig,bbox_inches="tight")
    try:
        from IPython import get_ipython
        from IPython.display import display, Image
        if get_ipython() is not None:
            display(Image(filename=str(NOTEBOOK_IMAGES[-1])))
    except ImportError:
        pass
    plt.close(fig)

bundle_path = output_path("figures/evaluation_figures.pdf")
with PdfPages(bundle_path) as pdf:
    fig, axes = plt.subplots(1,2,figsize=(12.8,4.7),layout="constrained")
    for j,model in enumerate(MODELS):
        for k,metric in enumerate(("macro_f1","accuracy")):
            pos=k+(j-1)*0.18
            met=RESULTS["all"][model][metric]*100
            lo,hi=CI["all"][model][metric]
            axes[0].plot([pos,pos],[lo*100,hi*100],color=COLORS[model],lw=2)
            axes[0].scatter(pos,met,color=COLORS[model],s=55,label="Modèle "+model if k==0 else None,zorder=3)
            axes[0].annotate(f"{met:.2f}",(pos,hi*100),xytext=(0,6),textcoords="offset points",ha="center",fontsize=9)
        f1=[RESULTS["all"][model]["per_class"][label]["f1"]*100 for label in LABELS]
        axes[1].bar(np.arange(3)+(j-1)*0.23,f1,width=.21,color=COLORS[model],label="Modèle "+model)
    axes[0].set(xticks=[0,1],xticklabels=["Macro-F1","Accuracy"],ylabel="Score (%)",ylim=(40,70),
                title="Performance globale et IC à 95 %")
    axes[0].legend(frameon=False,loc="lower right")
    axes[1].set(xticks=range(3),xticklabels=FR_LABELS,ylabel="F1 (%)",ylim=(0,100),title="F1 par sentiment")
    for ax in axes: ax.grid(axis="y",alpha=.18); ax.set_axisbelow(True)
    fig.suptitle("AfriSenti ary · 2 961 commentaires · Évaluation exploratoire",fontsize=14)
    save_figure(fig,"01_global_and_class_metrics",pdf)

    fig,axes=plt.subplots(1,3,figsize=(14,4.8),layout="constrained")
    vmax=max(np.asarray(RESULTS["all"][m]["confusion"]).max() for m in MODELS)
    for ax,model in zip(axes,MODELS):
        matrix=np.array(RESULTS["all"][model]["confusion"])
        image_plot=ax.imshow(matrix,cmap="Blues",vmin=0,vmax=vmax)
        for i in range(3):
            for j in range(3):
                count=matrix[i,j];fraction=count/matrix[i].sum()
                ax.text(j,i,f"{count}\n{100*fraction:.1f} %",ha="center",va="center",
                        color="white" if count>vmax*.55 else "#202832",fontsize=10)
        ax.set(xticks=range(3),yticks=range(3),xticklabels=FR_LABELS,yticklabels=FR_LABELS,
               xlabel="Sentiment prédit",ylabel="Sentiment réel",title="Modèle "+model)
    fig.colorbar(image_plot,ax=axes,shrink=.8,label="Nombre de commentaires",pad=.02)
    fig.suptitle("Matrices de confusion · Effectifs et pourcentages par ligne",fontsize=14)
    save_figure(fig,"02_confusion_matrices",pdf)

    shown=["arabic","latin","mixed","arabizi_proxy"]
    fig,axes=plt.subplots(1,2,figsize=(13.2,5.6),layout="constrained")
    for ax,metric,title in zip(axes,("macro_f1","negative_recall"),
                              ("Macro-F1 et IC bootstrap","Rappel négatif et IC de Wilson")):
        for j,model in enumerate(MODELS):
            for i,group in enumerate(shown):
                value=RESULTS[group][model]["macro_f1"] if metric=="macro_f1" else RESULTS[group][model]["per_class"]["negative"]["recall"]
                lo,hi=CI[group][model][metric];pos=i+(j-1)*.18
                if lo is not None:ax.plot([lo*100,hi*100],[pos,pos],color=COLORS[model],lw=1.7)
                ax.scatter(value*100,pos,color=COLORS[model],s=35,label="Modèle "+model if i==0 else None,zorder=3)
        labels=[GROUP_NAMES[g]+"\n"+("n="+str(len(GROUPS[g])) if metric=="macro_f1" else "négatifs="+str(RESULTS[g]["05"]["per_class"]["negative"]["support"]))
                +(" · prudence" if group_warnings(Y[GROUPS[g]]) else "") for g in shown]
        ax.set(yticks=range(len(shown)),yticklabels=labels,xlabel="Score (%)",xlim=(0,100),title=title)
        ax.invert_yaxis();ax.grid(axis="x",alpha=.18)
    axes[0].legend(frameon=False,loc="lower right")
    fig.suptitle("Écritures distinctes · *Proxy Arabizi non vérifié, groupes recouvrants",fontsize=13)
    save_figure(fig,"03_script_and_proxy_metrics",pdf)

    shown=["all","arabic","latin","mixed","arabizi_proxy"]
    fig,axes=plt.subplots(1,2,figsize=(13.2,5.8),layout="constrained")
    for ax,key,prefix,title in zip(axes,("delta_macro_f1","delta_negative_recall"),
                                  ("macro","negative"),("Différence de Macro-F1","Différence de rappel négatif")):
        for j,comparison in enumerate(("05-03","05-04")):
            color=("#0072B2","#009E73")[j]
            for i,group in enumerate(shown):
                row=next(r for r in COMPARISONS if r["group"]==group and r["comparison"]==comparison)
                pos=i+(j-.5)*.23;lo,hi=row[prefix+"_ci_low"],row[prefix+"_ci_high"]
                if lo is not None:ax.plot([100*lo,100*hi],[pos,pos],color=color,lw=2)
                if row[key] is not None:ax.scatter(row[key]*100,pos,color=color,s=36,label=comparison if i==0 else None,zorder=3)
        ax.axvline(0,color="#333333",lw=1,ls="--")
        ax.set(yticks=range(len(shown)),yticklabels=[GROUP_NAMES[g]+(" · prudence" if group_warnings(Y[GROUPS[g]]) else "") for g in shown],
               xlabel="Écart en points de pourcentage",title=title)
        ax.invert_yaxis();ax.grid(axis="x",alpha=.15)
    axes[0].legend(frameon=False,loc="lower right")
    axes[0].get_legend().remove()
    fig.legend(*axes[0].get_legend_handles_labels(),loc="outside lower center",ncol=2,frameon=False)
    fig.suptitle("Comparaisons appariées · 2 000 tirages · IC à 95 % exploratoires",fontsize=13)
    save_figure(fig,"04_paired_differences",pdf)

    fig,ax=plt.subplots(figsize=(9,4.8),layout="constrained")
    categories=["Labels réels"]+["Modèle "+m for m in MODELS]
    counts=[np.bincount(Y,minlength=3)]+[np.bincount(PRED[m],minlength=3) for m in MODELS]
    bottom=np.zeros(4)
    for k,label in enumerate(FR_LABELS):
        heights=np.array([row[k]/sum(row)*100 for row in counts])
        ax.bar(categories,heights,bottom=bottom,color=CLASS_COLORS[k],label=label,width=.62)
        for i,v in enumerate(heights):ax.text(i,bottom[i]+v/2,f"{v:.1f} %",ha="center",va="center",color="white")
        bottom+=heights
    ax.set(ylabel="Proportion (%)",ylim=(0,100),title="Distribution réelle et distribution des prédictions")
    ax.legend(loc="upper center",bbox_to_anchor=(.5,-.1),ncol=3,frameon=False)
    save_figure(fig,"05_prediction_distribution",pdf)

    fig,ax=plt.subplots(figsize=(9,4.8),layout="constrained")
    splits=["train","dev","test"];bottom=np.zeros(3)
    for k,group in enumerate(("arabic","latin","mixed","other")):
        heights=np.array([SCRIPT_COUNTS[s].get(group,0)/len(DATA[s])*100 for s in splits])
        ax.bar(splits,heights,bottom=bottom,color=SCRIPT_COLORS[k],label=GROUP_NAMES[group],width=.6)
        for i,v in enumerate(heights):
            if v>6:ax.text(i,bottom[i]+v/2,f"{v:.1f} %",ha="center",va="center",color="white" if k!=1 else "#202832")
        bottom+=heights
    ax.set(xticks=range(3),xticklabels=[s+"\nn="+str(len(DATA[s])) for s in splits],
           ylabel="Proportion (%)",ylim=(0,100),title="Composition des splits après neutralisation des mentions et URL")
    ax.legend(loc="upper center",bbox_to_anchor=(.5,-.14),ncol=4,frameon=False)
    save_figure(fig,"06_script_distribution",pdf)
ARTIFACTS.append(bundle_path)
composition=[{"split":s,"group":g,"count":SCRIPT_COUNTS[s].get(g,0),
              "proportion":SCRIPT_COUNTS[s].get(g,0)/len(DATA[s])}
             for s in ("train","dev","test") for g in ("arabic","latin","mixed","other")]
write_csv("tables/script_composition.csv",composition,list(composition[0]))
print("Six figures exported to PNG/SVG/PDF; combined PDF:",bundle_path)

## 7. Rapport, recommandation et traçabilité

Le rapport distingue les gains globaux, les régressions par groupe et l'incertitude. La recommandation est une décision expérimentale, pas une validation de déploiement.

Le protocole Hespress est fourni dans `docs/datasets/HESPRESS_BENCHMARK_PROTOCOL.md`. Le jeu non annoté existant ne devient pas une vérité terrain par l'exécution de ce notebook.

La dernière cellule vérifie que tous les fichiers d'entrée protégés sont inchangés. Une exécution échouée conserve son dossier partiel ; relancer depuis la première cellule crée un nouveau run et n'écrase pas le précédent.

In [ ]:
def pct(value):
    return "—" if value is None else f"{100*value:.2f} %"

def interval(low, high, points=False):
    if low is None:
        return "non évaluable"
    return f"[{100*low:+.2f} ; {100*high:+.2f}] points" if points else f"[{100*low:.2f} ; {100*high:.2f}] %"

def md_table(headers, rows):
    def escape(value):
        return str(value).replace("|",r"\|").replace("\n"," ")
    lines=["| "+" | ".join(map(escape,headers))+" |",
           "| "+" | ".join("---" for _ in headers)+" |"]
    lines += ["| "+" | ".join(map(escape,row))+" |" for row in rows]
    return "\n".join(lines)

main_comparison=next(r for r in COMPARISONS if r["group"]=="all" and r["comparison"]=="05-03")
proxy_comparison=next(r for r in COMPARISONS if r["group"]=="arabizi_proxy" and r["comparison"]=="05-03")
if main_comparison["macro_ci_low"] <= 0 <= main_comparison["macro_ci_high"]:
    recommendation=("Conserver le modèle 03 comme référence de comparaison principale et le modèle 05 comme "
        "challenger documenté. Le gain global de 05 face à 03 reste incertain. Les deux artefacts doivent être "
        "conservés ; 05 ne doit pas être promu automatiquement comme modèle supérieur.")
else:
    recommendation=("Le changement global est détectable dans cet intervalle exploratoire, mais la décision "
        "doit intégrer les sous-groupes et une évaluation indépendante. Conserver les références existantes.")
if proxy_comparison["delta_negative_recall"] is not None and proxy_comparison["delta_negative_recall"] < 0:
    recommendation += (" Le rappel négatif du proxy Arabizi diminue avec 05 ; ce signal ne constitue pas une "
                       "mesure linguistique vérifiée et son intervalle doit être examiné.")


def comparison_row(group,baseline):
    return next(r for r in COMPARISONS if r["group"]==group and r["comparison"]=="05-"+baseline)

def class_delta(label,metric,baseline="03"):
    return 100*(RESULTS["all"]["05"]["per_class"][label][metric]
                -RESULTS["all"][baseline]["per_class"][label][metric])

reading = [
    f"Face à 03, le F1 négatif évolue de {class_delta('negative','f1'):+.2f} point, "
    f"contre {class_delta('neutral','f1'):+.2f} pour le neutre et "
    f"{class_delta('positive','f1'):+.2f} pour le positif. Le rappel négatif global "
    f"varie de {class_delta('negative','recall'):+.2f} point, mais la précision négative "
    f"de {class_delta('negative','precision'):+.2f} : davantage de vrais négatifs ne "
    "signifie pas nécessairement un meilleur F1 négatif.",
    f"Sur l'arabe seul, le rappel négatif gagne "
    f"{100*comparison_row('arabic','03')['delta_negative_recall']:+.2f} points face à 03 "
    f"(IC {interval(comparison_row('arabic','03')['negative_ci_low'],comparison_row('arabic','03')['negative_ci_high'],True)}). "
    f"Il évolue de {100*comparison_row('latin','03')['delta_negative_recall']:+.2f} points "
    f"sur le latin et de {100*proxy_comparison['delta_negative_recall']:+.2f} "
    "sur le proxy Arabizi. Les intervalles de ces deux derniers écarts incluent ou "
    "touchent zéro : les baisses observées ne sont pas des preuves confirmatoires.",
    f"Face à 04, le gain global de Macro-F1 est de "
    f"{100*comparison_row('all','04')['delta_macro_f1']:+.2f} points, avec un gain "
    f"de {100*comparison_row('latin','04')['delta_macro_f1']:+.2f} sur le latin. "
    f"Cependant, le rappel négatif en arabe seul baisse de "
    f"{100*comparison_row('arabic','04')['delta_negative_recall']:.2f} points "
    f"(IC {interval(comparison_row('arabic','04')['negative_ci_low'],comparison_row('arabic','04')['negative_ci_high'],True)}). "
    "Le gain global masque donc un compromis entre sous-groupes.",
    "Le groupe mixte ne compte que 75 textes, dont 15 négatifs. Ses résultats "
    "restent fragiles. Les écritures détectées ne prouvent ni la langue ni "
    "l'absence d'artefacts de source ; les cas illustratifs le montrent."
]
lines=["# LahjaLens — Évaluation robuste des modèles 03, 04 et 05","",
       f"Run d'évaluation : **{RUN_ID}**. Prédictions sources : **{REFERENCE_RUN}**.","",
       "**Résultats exploratoires : le test AfriSenti a déjà influencé le développement. "
       "Aucun entraînement, aucune modification de modèle ou de dataset, aucun push Git.**","",
       "## Conclusion expérimentale","",recommendation,"",

       "## Lecture des gains et régressions","",
       *[part for paragraph in reading for part in (paragraph,"")],
       "## Provenance et opérations exécutées","",
       f"Comparaison sur **{len(IDS)} identifiants strictement identiques**, avec labels vérifiés. "
       "Les matrices et métriques globales reproduisent le rapport source. Le bootstrap apparié historique "
       "est reproduit avec 2 000 tirages et la même graine.","",
       "Exécuté : lecture et empreintes des artefacts, statistiques NumPy, analyse des écritures, "
       "bootstrap, dépistage des quasi-doublons, graphiques et consultations HDFS indiquées ci-dessous.","",
       "Non exécuté : entraînement, nouvelle inférence Spark, exécution d'un notebook historique, "
       "nouveau fsck HDFS, annotation humaine, enregistrement MLflow, intégration Silver/Gold ou déploiement.","",
       md_table(["Dépendance","Version"],[(k,v) for k,v in VERSIONS.items()]),"",
       md_table(["Modèle","Disponibilité HDFS","Version Spark des métadonnées"],
                [(m,MODEL_METADATA[m]["status"],MODEL_METADATA[m].get("spark_version","non vérifiée")) for m in MODELS]),"",
       "Les empreintes HDFS ci-dessus portent sur les métadonnées, pas sur tous les poids. "
       "Une indisponibilité HDFS ne transforme pas le calcul local des métriques en nouvelle validation du modèle.","",
       "## Résultats globaux","",
       md_table(["Modèle","Accuracy","Macro-F1","IC 95 % Macro-F1"],
          [(m,pct(RESULTS["all"][m]["accuracy"]),pct(RESULTS["all"][m]["macro_f1"]),
            interval(*CI["all"][m]["macro_f1"])) for m in MODELS]),"",
       "![Scores globaux et par classe](figures/01_global_and_class_metrics.png)","",
       "## Précision, rappel et F1 par classe","",
       md_table(["Modèle","Classe","Support","Précision","Rappel","F1"],
          [(m,label,RESULTS["all"][m]["per_class"][label]["support"],
            pct(RESULTS["all"][m]["per_class"][label]["precision"]),
            pct(RESULTS["all"][m]["per_class"][label]["recall"]),
            pct(RESULTS["all"][m]["per_class"][label]["f1"])) for m in MODELS for label in LABELS]),"",
       "## Matrices et distribution des prédictions","",
       "Les matrices affichent les classes réelles en lignes et les classes prédites en colonnes.","",
       "![Matrices de confusion](figures/02_confusion_matrices.png)","",
       "![Distribution des prédictions](figures/05_prediction_distribution.png)","",
       "## Écritures et proxy Arabizi","",
       "Mentions et URL neutralisées uniquement pour l'analyse. Latin ne signifie pas Arabizi. "
       "Le proxy numérique v1 peut manquer les textes sans chiffres et inclure à tort des termes tels que COVID19. "
       "Il recoupe les groupes latin/mixte et ne doit pas être additionné à ceux-ci.","",
       md_table(["Groupe","n","Négatifs","Macro-F1 03","Macro-F1 04","Macro-F1 05","Rappel négatif 05","IC rappel 05","Prudence"],
          [(GROUP_NAMES[g],len(GROUPS[g]),RESULTS[g]["05"]["per_class"]["negative"]["support"],
            pct(RESULTS[g]["03"]["macro_f1"]),pct(RESULTS[g]["04"]["macro_f1"]),pct(RESULTS[g]["05"]["macro_f1"]),
            pct(RESULTS[g]["05"]["per_class"]["negative"]["recall"]) if RESULTS[g]["05"]["per_class"]["negative"]["support"] else "—",
            interval(*CI[g]["05"]["negative_recall"]),"; ".join(group_warnings(Y[GROUPS[g]])) or "aucune alerte d'effectif")
           for g in ("arabic","latin","mixed","other","arabizi_proxy")]),"",
       "Les seuils d'alerte sont n<100 ou support d'une classe<30. Ils ne garantissent pas la fiabilité "
       "des groupes dépassant ces seuils. Les scores du groupe mixte restent notamment fragiles.","",
       "![Résultats par écriture](figures/03_script_and_proxy_metrics.png)","",
       "![Composition des splits](figures/06_script_distribution.png)","",
       "## Comparaisons appariées","",
       md_table(["Groupe","Comparaison","Écart Macro-F1","IC 95 %","Écart rappel négatif","IC 95 %"],
          [(GROUP_NAMES[r["group"]],r["comparison"],f'{100*r["delta_macro_f1"]:+.2f} points',
            interval(r["macro_ci_low"],r["macro_ci_high"],True),
            "—" if r["delta_negative_recall"] is None else f'{100*r["delta_negative_recall"]:+.2f} points',
            interval(r["negative_ci_low"],r["negative_ci_high"],True)) for r in COMPARISONS]),"",
       "Les tirages sont appariés entre modèles. Les intervalles supposent des lignes indépendantes et "
       "ne corrigent ni l'exposition antérieure au test, ni la multiplicité, ni les dépendances de conversation inconnues.","",
       "![Différences appariées](figures/04_paired_differences.png)","",
       "## Exemples en désaccord et erreurs fréquentes","",
       md_table(["Comparaison","Groupe","Désaccords","Corrigés par 05","Nouvelles erreurs de 05","Deux erreurs différentes"],
          [(r["comparison"],GROUP_NAMES[r["group"]],r["disagreements"],r["corrected_by_05"],
            r["new_errors_by_05"],r["both_wrong_different"]) for r in DISAGREEMENT_SUMMARIES
           if r["group"] in ("all","arabizi_proxy","mixed")]),"",
       md_table(["Modèle","Vrai → prédit","Nombre"],
          [(m,r["true_class"]+" → "+r["predicted_class"],r["count"]) for m in MODELS
           for r in [x for x in ERROR_COUNTS if x["model"]==m and x["group"]=="all"][:3]]),"",

       md_table(["Comparaison","Classe réelle","Corrigés par 05","Nouvelles erreurs","Solde correct"],
          [(r["comparison"],r["true_class"],r["corrected_by_05"],r["new_errors_by_05"],
            f'{r["net_correct"]:+d}') for r in CLASS_CHANGES]),"",
       "Les transitions complètes, ventilées par écriture et classe réelle, figurent dans "
       "tables/disagreement_transitions.csv. Elles décomposent les gains et pertes sans "
       "attribuer de cause aux marqueurs textuels.","",
       md_table(["Comparaison","Identifiant","Label","Avant → 05","Note exploratoire"],
          [(r["comparison"],r["record_id"],r["true_class"],
            r["baseline_prediction"]+" → "+r["prediction_05"],r["review_hypothesis"])
           for r in CASE_SUMMARIES]),"",
       "Ces cas sont illustratifs, sans estimation de fréquence. Les textes et les notes "
       "de lecture sont dans [le dossier local des cas](private_review/disagreement_case_notes.md). "
       "Les hypothèses ne sont ni des annotations linguistiques vérifiées ni des explications causales du modèle.","",
       "Les identifiants complets sont dans tables/disagreement_identifiers.csv. Les textes et un panel "
       "de relecture déterministe sont dans private_review/ ; ils ne doivent pas être publiés automatiquement. "
       "Les champs de cause et de langue humaines restent vides. La présence d'un marqueur ne démontre pas une cause d'erreur.","",
       "## Reproductibilité et limites confirmées","",
       f"- Recouvrements exacts normalisés entre splits : {CHECKS['exact_normalized_overlap']}.",
       f"- Quasi-doublons détectés : **{len(near_rows)} paires**, recherche approximative à seuil 0,85. "
       "Les identifiants et similarités sont dans tables/near_duplicates.csv.",
       "- Le split interne original est reproduit exactement, mais ne regroupe pas les familles de quasi-doublons.",
       "- La préparation NFKC du notebook 01 n'est pas embarquée de la même manière dans le modèle 05. "
       "Un contrat Silver explicite et des tests d'équivalence restent nécessaires.",

       "- Le modèle 03 requiert la création externe de char_tokens par add_features/char_ngrams ; "
       "ces caractéristiques ne se reconstruisent pas avec un simple comment_text dans son pipeline sauvegardé. "
       "Le notebook 04 déclare ses transformations de caractères en SQL dans le pipeline. "
       "Ces constats sont statiques : aucune équivalence d'inférence complète n'est affirmée.",
       "- Les appels historiques à overwrite() sont seulement inspectés. Le répertoire afrisenti_v1 "
       "du notebook 02 contient aussi des expériences ; il ne doit pas être réécrit.",
       "- Les sources utilisent une référence Git mutable ; les empreintes locales sont vérifiées, "
       "mais la version amont doit être figée pour une nouvelle reproduction.",
       "- Le lecteur TSV strict signale les anomalies sans réinterpréter ni modifier les données préparées.",
       "- Les probabilités ne sont pas présentes dans ces CSV de prédictions : calibration, log-loss "
       "et décisions d'abstention ne sont pas évaluées.","",
       "Détails structurés : [contrôles](reproducibility_checks.json), [résultats](evaluation.json) "
       "et [manifeste](run_manifest.json).","",
       "## Benchmark Hespress","",
       f"État observé de l'échantillon : {CHECKS['hespress_available_sample']}.","",
       "Il ne constitue pas un benchmark de sentiment indépendant disponible. Le protocole prévoit "
       "une séparation par articles et familles de doublons, deux annotations aveugles, arbitrage, "
       "un lot représentatif et un lot défi Arabizi/mixte séparés. Les labels finaux restent réservés.","",
       "Protocole du dépôt : docs/datasets/HESPRESS_BENCHMARK_PROTOCOL.md.","",
       "## Fichiers produits","",
       "- tables/ : métriques, matrices, distributions, erreurs, désaccords, bootstrap et quasi-doublons.",
       "- figures/ : six figures PNG/SVG/PDF et evaluation_figures.pdf.",
       "- private_review/ : exemples textuels destinés uniquement à la relecture locale.",
       "- evaluation.json et run_manifest.json : résultats et provenance reproductible.",""]
report_path = output_path("comparison_report.md")
report_path.write_text("\n".join(lines),encoding="utf-8")
ARTIFACTS.append(report_path)

# Recheck every protected input after all computations and HDFS reads.
changed = [relative for relative,digest in INPUT_HASHES.items() if sha256(ROOT/relative)!=digest]
assert not changed, "Inputs changed during evaluation: " + repr(changed)
CHECKS["input_files_unchanged_at_end"] = {"status":"passed","files":len(INPUT_HASHES)}
evaluation = {"status":"completed","run_id":RUN_ID,"reference_run":REFERENCE_RUN,
    "selection_or_training_performed":False,"historical_test_already_consulted":True,
    "labels":LABELS,"metrics":RESULTS,"intervals":CI,"paired_comparisons":COMPARISONS,
    "disagreement_summary":DISAGREEMENT_SUMMARIES,"recommendation":recommendation,
    "checks":CHECKS,"versions":VERSIONS,"bootstrap_repetitions":BOOTSTRAPS,"seed":SEED}
write_json("evaluation.json",evaluation)
write_json("run_manifest.json",{"status":"completed","run_id":RUN_ID,
    "created_at":datetime.now(timezone.utc).isoformat(),"elapsed_seconds":round(time.monotonic()-STARTED,2),
    "versions":VERSIONS,"input_sha256":INPUT_HASHES,"model_metadata":MODEL_METADATA,
    "source_notebook_sha256":sha256(SOURCE_NOTEBOOK),"output_directory":str(OUT_DIR),
    "output_sha256":{str(p.relative_to(OUT_DIR)):sha256(p) for p in ARTIFACTS},
    "input_files_unchanged":True,
    "executed_notebook_note":"The CLI runner adds an executed notebook copy after this manifest; no historical notebook is executed."})
print("COMPLETED",RUN_ID)
print("REPORT",report_path)
print("REFERENCE RECOMMENDATION",recommendation)
print("INPUT FILES UNCHANGED",len(INPUT_HASHES))